# Fine-tune GPT-2 Vietnamese for Math Word Problems

**Goal:** fine-tune `NlpHUST/gpt2-vietnamese` to solve Vietnamese math word problems.

**Inputs:** `train.json`, `valid.json`, and the local GPT-2 model folder from Kaggle Input.

**Pipeline:** load data → SFT training → generate validation outputs → evaluate by relative error.

**Rules:** Internet OFF, no extra data/API/LLM, total runtime ≤ 3 hours.


In [1]:
import os, sys, json, math, time, re, random, hashlib, inspect, unicodedata
from collections import Counter
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


Torch: 2.10.0+cu128
CUDA : True | GPU count: 1
0 Tesla T4


In [8]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"      # Phase 2

# Run mode: "phase1" -> infer on valid;  "phase2" -> infer on test
RUN_MODE = "phase1"

PROMPT_TEMPLATE = "Hãy giải bài toán sau, trình bày ngắn gọn từng bước rồi viết 'Đáp án là: <số>' ở cuối.\nBài toán: {q}\nLời giải: "
ANSWER_SUFFIX_TEMPLATE = "\nĐáp án là: {a}"
SAFE_EOS_ID = 50256

OUTPUT_DIR = Path("/kaggle/working/gpt2_math_ckpt_v2")
VALID_OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
VALID_REPORT_PATH = Path("/kaggle/working/valid_report.json")
TEST_OUTPUT_PATH = Path("/kaggle/working/test_predictions.json")
BASELINE_OUTPUT_PATH = Path("/kaggle/working/baseline_valid_output.json")
BASELINE_REPORT_PATH = Path("/kaggle/working/baseline_valid_report.json")

MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None

FILTER_DUPLICATE_QUERIES = False
DROP_NON_EXTRACTABLE = True          
NORMALIZE_TARGET_FORMAT = True       # force "...\nĐáp án là: <num>" 
KEEP_ORIGINAL_REASONING = True       # keep original solution text, only fix the tail


EPOCHS = 1
N_POSITIONS = 1024
MAX_LENGTH = 768 # Old: 512
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM = 8
LR = 5e-5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 42

MAX_NEW_TOKENS = 320 # Old: 256
RUN_BASELINE_FIRST = False

# Decoding
DECODE_MODE = "beam" # "greedy" | "beam" | "self_consistency"
NUM_BEAMS = 4
NO_REPEAT_NGRAM = 4
REPETITION_PENALTY = 1.2
LENGTH_PENALTY = 0.9
# self-consistency only
SC_N = 5
SC_TEMPERATURE = 0.7
SC_TOP_P = 0.9

# Inference safety knobs
INFER_FP16 = True       # half precision at inference

USE_TYPE_AWARE_FEWSHOT = False      # enable simple few-shot prepend per type at inference


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("TRAIN_FILE :", TRAIN_FILE)
print("VALID_FILE :", VALID_FILE)
print("MODEL_NAME :", MODEL_NAME)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("RUN_MODE   :", RUN_MODE)


TRAIN_FILE : /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_FILE : /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
MODEL_NAME : /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
OUTPUT_DIR : /kaggle/working/gpt2_math_ckpt_v2
RUN_MODE   : phase1


In [9]:
# ============================================================
# 1.5. Tokenizer smoke-check
# ============================================================
def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # The provided tokenizer maps id 50256 to a normal token string ("hue")
    # even though the task requires using 50256 as EOS/PAD. Strip it manually.
    return tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

probe = "Bài toán: 2+3=?\nLời giải: "
print("Vocab size:", tokenizer.vocab_size)
print("Token ids :", tokenizer(probe, add_special_tokens=False)["input_ids"][:20], "...")
print("SAFE_EOS raw decode:", repr(tokenizer.decode([SAFE_EOS_ID])))
print("SAFE_EOS stripped  :", repr(decode_model_text(tokenizer, [16, SAFE_EOS_ID])))


Vocab size: 50257
Token ids : [3108, 1329, 30, 416, 15, 23, 33, 35, 203, 6501, 800, 30, 225] ...
SAFE_EOS raw decode: 'hue'
SAFE_EOS stripped  : ','


In [10]:
# ============================================================
# 2. Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records))
print("sample query:", train_records[0]["query_vi"][:200])


train: 100000 | valid: 1000
sample query: Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực


In [11]:
# ============================================================
# 3. Answer extraction + data cleaning
# ============================================================
RE_ANCHORS_VI = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
]
RE_ANCHORS_EN = [
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
RE_BOXED = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

# Numeric literal patterns
RE_NUM_VI_DEC = re.compile(r"-?\d+,\d+")          # 1,5
RE_NUM_EN_DEC = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")
RE_NUM_LOOSE  = re.compile(r"-?\d+(?:[.,]\d+)?")

def _clean_tail(s: str) -> str:
    s = s.strip()
    # take first non-empty line after anchor
    s = s.split("\n", 1)[0].strip()
    # strip trailing punctuation / currency
    s = re.sub(r"[.,;:。、,]+$", "", s)
    # drop "đô la", "USD", "%" trailing units
    s = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)\b.*$", "", s, flags=re.IGNORECASE)
    s = s.strip()
    return s

def extract_anchor_answer(text: str | None) -> str | None:
    if not text:
        return None
    best_pos, best_tail = -1, None
    for pat in RE_ANCHORS_VI + RE_ANCHORS_EN:
        for m in pat.finditer(text):
            if m.end() > best_pos:
                best_pos = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    # boxed fallback (take LAST boxed)
    boxes = RE_BOXED.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(s: str | None) -> float | None:
    if s is None:
        return None
    t = s.strip()
    if not t:
        return None
    # pure VI decimal
    if RE_NUM_VI_DEC.fullmatch(t):
        try:
            v = float(t.replace(",", "."))
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    if RE_NUM_EN_DEC.fullmatch(t):
        try:
            v = float(t)
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    # strip thousand-sep commas then try leading number
    cleaned = t.replace(",", "")
    m = RE_NUM_EN_DEC.search(cleaned)
    if m:
        try:
            v = float(m.group())
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    # fallback: any number-like
    m = RE_NUM_LOOSE.search(t)
    if m:
        try:
            v = float(m.group().replace(",", "."))
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    return None

def extract_gold(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("response_vi"))
    return s, parse_number(s)

def extract_pred(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("model_output"))
    return s, parse_number(s)


# ---- normalize target: keep reasoning, force final anchor ----
RE_TRAILING_ANCHORS = re.compile(
    r"(\s*(####\s*[-\d., ]*|"
    r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?[^\n]*|"
    r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?[^\n]*|"
    r"the\s*answer\s*is\s*[:：]?[^\n]*))+\s*$",
    re.IGNORECASE,
)

def normalize_response(resp: str, gold_num: float | None, gold_str: str | None) -> str:
    """Strip trailing answer-anchors, then append a single canonical anchor."""
    body = RE_TRAILING_ANCHORS.sub("", resp.rstrip()).rstrip()
    # decide the canonical answer string
    if gold_num is not None:
        if gold_num == int(gold_num):
            canonical = str(int(gold_num))
        else:
            # use the original gold string if it looks well-formed, else float repr
            canonical = gold_str if gold_str and re.fullmatch(r"-?\d+(?:[.,]\d+)?", gold_str) else f"{gold_num:g}"
    else:
        canonical = gold_str if gold_str else ""
    return f"{body}\nĐáp án là: {canonical}"

def clean_records(records: list[dict], split: str) -> list[dict]:
    seen = set()
    out = []
    dropped_no_ans = 0
    dropped_dup = 0
    normalized = 0

    for rec in records:
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_no_ans += 1
            continue
        if FILTER_DUPLICATE_QUERIES and split == "train":
            key = q
            if key in seen:
                dropped_dup += 1
                continue
            seen.add(key)

        gold_str, gold_num = extract_gold(rec)
        if DROP_NON_EXTRACTABLE and split == "train" and gold_num is None:
            dropped_no_ans += 1
            continue

        new_r = r
        if NORMALIZE_TARGET_FORMAT and split == "train" and KEEP_ORIGINAL_REASONING:
            new_r = normalize_response(r, gold_num, gold_str)
            if new_r != r:
                normalized += 1

        out.append({
            **rec,
            "query_vi": q,
            "response_vi": new_r,
            "_gold_num": gold_num,
            "_gold_str": gold_str,
        })

    print(f"[{split}] kept={len(out)} dropped_dup={dropped_dup} "
          f"dropped_no_ans={dropped_no_ans} normalized={normalized}")
    return out

train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid")  # no filtering, just gold extraction

print("\nExample BEFORE / AFTER:")
ex = train_records[0]
print("BEFORE:", ex["response_vi"][-200:])
print("AFTER :", train_clean[0]["response_vi"][-200:])


[train] kept=99697 dropped_dup=0 dropped_no_ans=303 normalized=99697
[valid] kept=1000 dropped_dup=0 dropped_no_ans=0 normalized=0

Example BEFORE / AFTER:
BEFORE: làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây. ####1200 Đáp án là: 1200
AFTER :  vụ luôn làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây.
Đáp án là: 1200


In [12]:
# ============================================================
# 4. SFT dataset and collator
# ============================================================
class SFTDataset(Dataset):
    """Tokenize (prompt, response), mask loss on prompt + padding."""

    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"])
        response = rec["response_vi"]

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        budget = self.max_length - len(p_ids)
        if budget <= 4:
            # prompt too long -> truncate from left of prompt
            p_ids = p_ids[-(self.max_length - 8):]
            budget = self.max_length - len(p_ids)

        if len(r_ids) > budget:
            # keep the last `tail_keep` tokens (carries the anchor + answer + EOS)
            tail_keep = min(96, budget // 2)
            head_keep = budget - tail_keep
            r_ids = r_ids[:head_keep] + r_ids[-tail_keep:]

        ids = p_ids + r_ids
        labels = ([-100] * len(p_ids) + r_ids)

        # Defensive clamp: never feed out-of-range ids to embedding layer.
        ids = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

_ds_probe = SFTDataset(train_clean[:1], tokenizer, MAX_LENGTH)
_sample = _ds_probe[0]
print("sample len:", len(_sample["input_ids"]),
      "| n_loss_tokens:", sum(1 for x in _sample["labels"] if x != -100))
print("last 15 tokens:", decode_model_text(tokenizer, _sample["input_ids"][-15:]))


sample len: 239 | n_loss_tokens: 101
last 15 tokens:  * 8 = 1200 ngọn măng tây.
Đáp án là: 1200


In [13]:
# ============================================================
# 5. Evaluation utilities
# ============================================================
def rel_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(re_val, is_extractable):
    if not is_extractable or re_val is None:
        return 0
    if re_val <= 0.01: return 10
    if re_val <= 0.10: return 5
    if re_val <= 0.50: return 1
    return 0

def evaluate(pred_items, gold_items):
    assert len(pred_items) == len(gold_items), (len(pred_items), len(gold_items))
    rows = []
    total = 0
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in zip(pred_items, gold_items):
        gold_str, gold_num = extract_gold(gold_rec)
        pred_str, pred_num = extract_pred(pred_rec)

        is_extractable = pred_str is not None
        extractable += int(is_extractable)
        re_val = rel_error(pred_num, gold_num)
        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s
        buckets[s] = buckets.get(s, 0) + 1

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_str, "gold_num": gold_num,
            "pred_answer": pred_str, "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": n * 10,
            "score_10": total / n if n else 0.0,
            "score_pct": (total / (n * 10)) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }

def save_evaluation_report(pred_path, gold_records, report_path):
    pred_path = Path(pred_path); report_path = Path(report_path)
    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)
    result = evaluate(pred_items, gold_records)
    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Wrote {report_path}")
    return result


In [14]:
# ============================================================
# 6. StoppingCriteria + decoding
# ============================================================
class StopOnAnswerLine(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID,
                 patience_tokens: int = 24):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_answer = re.compile(r"đáp\s*án\s*l[àa]\s*[:：]\s*-?\d", re.IGNORECASE)
        self._matched_at = None     # token index when first matched

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        # decode only the generated tail (cheaper)
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 6:
            return False
        text = self.tok.decode(gen_tail, skip_special_tokens=True)
        m = self.re_answer.search(text)
        if m:
            if self._matched_at is None:
                self._matched_at = gen_tail.numel()
            # let it spit out a few more digits, then stop on newline or patience
            if "\n" in text[m.end():]:
                return True
            if gen_tail.numel() - self._matched_at >= self.patience:
                return True
        return False


In [15]:
# ============================================================
# 7. Greedy generation
# ============================================================
def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())

def _vote_answer(cands: list[str | None]) -> str | None:
    nums = []
    for c in cands:
        n = parse_number(extract_anchor_answer(c))
        if n is not None:
            nums.append((round(n, 6), c))
    if not nums:
        return cands[0] if cands else None
    counter = Counter(n for n, _ in nums)
    top_num, _ = counter.most_common(1)[0]
    for n, c in nums:
        if n == top_num:
            return c
    return cands[0]

@torch.inference_mode()
def generate_outputs(model_path_or_name, records, output_path,
                     max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[infer] Loading {model_path_or_name} on {device} (mode={mode}) ...", flush=True)

    tok = AutoTokenizer.from_pretrained(model_path_or_name, local_files_only=True)
    tok.pad_token_id = SAFE_EOS_ID
    tok.eos_token_id = SAFE_EOS_ID
    if tok.padding_side != "left":
        tok.padding_side = "left"   # left-pad for batched generation

    dtype = torch.float16 if (device == "cuda" and INFER_FP16) else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path_or_name, torch_dtype=dtype, local_files_only=True,
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()
    vocab_n = model.transformer.wte.num_embeddings

    outputs, t0 = [], time.time()
    for idx, rec in enumerate(tqdm(records, desc=f"gen[{mode}]")):
        prompt = build_prompt(rec)
        enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        ids = enc["input_ids"].clamp(max=vocab_n - 1)
        attn = enc.get("attention_mask")
        prompt_len = ids.shape[1]

        common_kwargs = dict(
            input_ids=ids, attention_mask=attn,
            max_new_tokens=max_new_tokens,
            pad_token_id=SAFE_EOS_ID, eos_token_id=SAFE_EOS_ID,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            repetition_penalty=REPETITION_PENALTY,
            stopping_criteria=StoppingCriteriaList([
                StopOnAnswerLine(tok, prompt_len=prompt_len)
            ]),
        )

        if mode == "greedy":
            gen = model.generate(do_sample=False, num_beams=1, **common_kwargs)
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "beam":
            gen = model.generate(
                do_sample=False, num_beams=NUM_BEAMS,
                length_penalty=LENGTH_PENALTY, early_stopping=True,
                **common_kwargs,
            )
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "self_consistency":
            cands = []
            for s in range(SC_N):
                gen = model.generate(
                    do_sample=True, temperature=SC_TEMPERATURE, top_p=SC_TOP_P,
                    num_beams=1, **common_kwargs,
                )
                cands.append(decode_model_text(tok, gen[0, prompt_len:]))
            text = _vote_answer(cands) or cands[0]
        else:
            raise ValueError(mode)

        outputs.append({
            "id": idx,
            "query_vi": rec["query_vi"],
            "type": rec.get("type"),
            "model_output": text,
        })

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)
    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"[infer] Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    # free
    del model
    torch.cuda.empty_cache()
    return outputs


In [16]:
# ============================================================
# 8. Optional baseline
# ============================================================
if RUN_BASELINE_FIRST:
    _ = generate_outputs(MODEL_NAME, valid_records, BASELINE_OUTPUT_PATH, MAX_NEW_TOKENS)
    baseline_result = save_evaluation_report(BASELINE_OUTPUT_PATH, valid_records, BASELINE_REPORT_PATH)
else:
    print("Skip baseline.")


Skip baseline.


In [17]:
# ============================================================
# 9. Train
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID
model.gradient_checkpointing_enable()
model.config.use_cache = False

train_ds = SFTDataset(train_clean, tokenizer, MAX_LENGTH)
# tiny eval sub-sample to keep eval fast
EVAL_SUB = min(200, len(valid_clean))
valid_ds = SFTDataset(valid_clean[:EVAL_SUB], tokenizer, MAX_LENGTH)
collator = PadCollator(pad_id=SAFE_EOS_ID)

eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
print(f"per_device_bs={PER_DEVICE_BATCH_SIZE} | grad_accum={GRAD_ACCUM} "
      f"| gpus={torch.cuda.device_count()} | eff_batch={eff_batch} "
      f"| train_size={len(train_ds)} | steps/epoch={steps_per_epoch}")

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = "epoch"
else:
    ta_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
)

t0 = time.time()
trainer.train()
train_dt = time.time() - t0
print(f"\n[train] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model_hash = sha256_dir(OUTPUT_DIR)
(OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
print("Saved checkpoint to:", OUTPUT_DIR, "| SHA256:", model_hash)

del trainer, model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


per_device_bs=4 | grad_accum=8 | gpus=2 | eff_batch=64 | train_size=99697 | steps/epoch=1558


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.225856,1.369177


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



[train] wall time: 8694.6s (144.91 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to: /kaggle/working/gpt2_math_ckpt_v2 | SHA256: 1003b34a4af2fc1bd7fde807f43bcb15584cd601396b6a59f3f6a8568b5d4cfd


In [18]:
# ============================================================
# 10. Inference on validation set
# ============================================================
if RUN_MODE == "phase1":
    valid_outputs = generate_outputs(
        OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE,
    )
    print("\nExample output:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

    valid_result = save_evaluation_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
    s = valid_result["summary"]
    print("\nFinal validation score:")
    print(f'{s["raw_score"]} / {s["max_raw_score"]}  ({s["score_pct"]*100:.2f}%)')
    print(f'Score /10: {s["score_10"]:.4f}')
    print("Buckets:", s["buckets"])
else:
    print("Skipping phase1 inference.")
    

[infer] Loading /kaggle/working/gpt2_math_ckpt_v2 on cuda (mode=beam) ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

gen[beam]:   0%|          | 0/1000 [00:00<?, ?it/s]

[infer] Wrote /kaggle/working/valid_output.json | 31.89 min | SHA256: ac703046f6acbd8b5cfe15e593e9f74e3839c766e49eeaba1f3d619c39202774

Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Nếu Susan đang chơi cờ bàn thì cô ấy sẽ di chuyển 2 ô từ ô đầu tiên đến ô cuối cùng, vì vậy cô ấy sẽ phải di chuyển tổng cộng 4 ô. Nếu cô ấy muốn di chuyển thêm 5 ô nữa thì cô ấy cần phải di chuyển thêm 6 ô nữa. Vì vậy, cô ấy sẽ cần di chuyển nhiều hơn 5 ô nữa. Tổng cộng, Susan cần di chuyển 12 + 6 + 7 + 8 + 9 + 10 + 11 + 12 + 13 + 14 + 15 + 16 + 18 + 19 + 20 + 21 + 22 + 23 + 24 + 25 + 26 + 27 + 28 + 29 + 30 + 31 + 3

In [19]:

# ============================================================
# 11. Error analysis preview
# ============================================================
if RUN_MODE == "phase1":
    rows = valid_result["rows"]
    by_type = {}
    for r in rows:
        t = r.get("type") or "UNK"
        by_type.setdefault(t, []).append(r)

    print("Per-type score breakdown:")
    print(f"{'type':<22s} {'n':>4s} {'mean':>6s} {'b10':>4s} {'b5':>4s} {'b1':>4s} {'b0':>4s} {'extr%':>6s}")
    for t, rs in sorted(by_type.items()):
        n = len(rs)
        mean_s = sum(r["score"] for r in rs) / n
        b10 = sum(r["score"] == 10 for r in rs)
        b5  = sum(r["score"] == 5 for r in rs)
        b1  = sum(r["score"] == 1 for r in rs)
        b0  = sum(r["score"] == 0 for r in rs)
        extr = 100 * sum(bool(r["extractable"]) for r in rs) / n
        print(f"{t:<22s} {n:>4d} {mean_s:>6.2f} {b10:>4d} {b5:>4d} {b1:>4d} {b0:>4d} {extr:>5.1f}%")

    bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
    print(f"\n{len(bad)} zero-score samples. Showing first 2:")
    for i, r in bad[:2]:
        pred = valid_outputs[i]; gold = valid_records[i]
        print("=" * 90)
        print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
        print("QUERY:", gold["query_vi"][:400])
        print("GOLD :", gold["response_vi"][-300:])
        print("PRED :", pred["model_output"][:800])


Per-type score breakdown:
type                      n   mean  b10   b5   b1   b0  extr%
GSM_AnsAug              209   0.59    6    7   29  167  86.6%
GSM_FOBAR               122   1.22   13    0   19   90  89.3%
GSM_Rephrased           197   0.43    2    4   44  147  92.4%
GSM_SV                   97   0.65    5    0   13   79  72.2%
MATH_AnsAug             173   0.65    8    2   22  141  82.1%
MATH_FOBAR               45   0.89    3    0   10   32  82.2%
MATH_Rephrased          116   0.56    5    0   15   96  91.4%
MATH_SV                  41   0.29    1    0    2   38  65.9%

790 zero-score samples. Showing first 2:
IDX: 0 | type: GSM_Rephrased | rel_error: None
QUERY: Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và gi

In [20]:
# ============================================================
# 12. Phase 2 — generate test_predictions.json
# ============================================================
if RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"Cannot find test.json at {TEST_FILE}")
    test_records = load_records(TEST_FILE)
    print("test:", len(test_records))

    test_outputs = generate_outputs(
        OUTPUT_DIR, test_records, TEST_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE,
    )

    # Required schema check
    required_keys = {"id", "query_vi", "type", "model_output"}
    missing = [k for k in required_keys if k not in test_outputs[0]]
    if missing:
        raise ValueError(f"Output missing keys: {missing}")

    print("First test prediction:")
    print(json.dumps(test_outputs[0], ensure_ascii=False, indent=2)[:1500])


In [21]:
# ============================================================
# 11. List saved artifacts
# ============================================================
WORKING_DIR  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
candidates = [
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    OUTPUT_DIR / "model_hash.txt",
    TEST_OUTPUT_PATH,
    Path(str(TEST_OUTPUT_PATH) + ".sha256.txt"),
]
for p in candidates:
    p = Path(p)
    print(p, "| exists =", p.exists(), "| size =", p.stat().st_size if p.exists() else None)

print("\nWorking dir contents:")
for p in sorted(Path(WORKING_DIR).glob("*")):
    print("-", p)


/kaggle/working/valid_output.json | exists = True | size = 926303
/kaggle/working/valid_report.json | exists = True | size = 237722
/kaggle/working/valid_output.json.sha256.txt | exists = True | size = 65
/kaggle/working/gpt2_math_ckpt_v2/model_hash.txt | exists = True | size = 65
/kaggle/working/test_predictions.json | exists = False | size = None
/kaggle/working/test_predictions.json.sha256.txt | exists = False | size = None

Working dir contents:
- /kaggle/working/.virtual_documents
- /kaggle/working/gpt2_math_ckpt_v2
- /kaggle/working/valid_output.json
- /kaggle/working/valid_output.json.sha256.txt
- /kaggle/working/valid_report.json
